<a href="https://colab.research.google.com/github/saitejamudapalli/Project-HealthCare-Provider-Analysis/blob/main/GOLD_LAYER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#PPCODE FOR GOLD_LAYER



from google.cloud import bigquery
from google.oauth2 import service_account
import pandas as pd
from datetime import datetime

# ===== CONFIG =====
PROJECT_ID = "even-blueprint-441418-p2"
SOURCE_DATASET = "SILVER_LAYER"
SOURCE_TABLE = "PATIENTS_SILVER"
TARGET_DATASET = "GOLD_LAYER"
TARGET_TABLE = "PATIENTS_GOLD"
KPI_TABLE = "GOLD_KPIS"
KEY_PATH = "/content/even-blueprint-441418-p2-043f8a9d855b.json(KEY).json"

# ===== READ SILVER =====
credentials = service_account.Credentials.from_service_account_file(KEY_PATH)
client = bigquery.Client(credentials=credentials, project=PROJECT_ID)

query = f"SELECT * FROM {PROJECT_ID}.{SOURCE_DATASET}.{SOURCE_TABLE}"
df_silver = client.query(query).to_dataframe()

# ===== ENRICHMENT =====
df_gold = df_silver.copy()

# Add utilization bucket if column exists
if "healthcare_expenses" in df_gold.columns:
    df_gold["expense_bucket"] = pd.cut(df_gold["healthcare_expenses"],
        bins=[-0.1, 0, 10000, 50000, 100000, 500000, float("inf")],
        labels=["0","<10k","10k-50k","50k-100k","100k-500k","500k+"] )

# Add has_geo flag
if "lat" in df_gold.columns and "lon" in df_gold.columns:
    df_gold["has_geo"] = df_gold["lat"].notnull() & df_gold["lon"].notnull()

# ===== KPIs =====
kpi_data = {
    "report_generated_utc": datetime.utcnow().isoformat(),
    "rows_in_silver": len(df_silver),
    "rows_in_gold": len(df_gold),
    "avg_expense": df_gold["healthcare_expenses"].mean() if "healthcare_expenses" in df_gold.columns else None,
    "avg_coverage": df_gold["healthcare_coverage"].mean() if "healthcare_coverage" in df_gold.columns else None,
    "geo_coverage_percent": round(df_gold["has_geo"].mean()*100, 2) if "has_geo" in df_gold.columns else None
}
df_kpi = pd.DataFrame([kpi_data])

# ===== UPLOAD TO BIGQUERY =====
table_ref_gold = f"{PROJECT_ID}.{TARGET_DATASET}.{TARGET_TABLE}"
table_ref_kpi = f"{PROJECT_ID}.{TARGET_DATASET}.{KPI_TABLE}"

job1 = client.load_table_from_dataframe(df_gold, table_ref_gold, job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
job1.result()

job2 = client.load_table_from_dataframe(df_kpi, table_ref_kpi, job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
job2.result()

print(f"✅ Gold layer loaded successfully to {table_ref_gold}")
print(f"✅ KPI table created at {table_ref_kpi}")